<a href="https://colab.research.google.com/github/sjagadee/py_torch_from_basics/blob/main/03_training_a_neuron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Breast Cancer Detection using a Simple Neural Network

In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

### Import Libraries
This cell imports all the necessary libraries for data manipulation, machine learning, and neural network operations.

## Load and basic clean up

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


### Load Data
This cell loads the breast cancer dataset from a URL into a pandas DataFrame and displays the first few rows.

In [ ]:
df.shape

(569, 33)

### Check Data Shape
This cell shows the number of rows and columns in the DataFrame.

In [ ]:
df.drop(columns=["id", "Unnamed: 32"], inplace=True)

### Drop Unnecessary Columns
This cell removes the 'id' column, which is an identifier and not useful for training, and 'Unnamed: 32', which is an empty column.

In [ ]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


### Display Cleaned Data
This cell displays the first few rows of the DataFrame after dropping the unnecessary columns.

## Train Test Split

In [ ]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Separate Features and Target
This cell splits the DataFrame into features (`X`) and the target variable (`y`). It then performs a train-test split to create training and testing datasets.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Standardize Features
This cell uses `StandardScaler` to standardize the features in both the training and testing sets. Standardization is crucial for many machine learning algorithms, especially neural networks, to perform optimally.

In [ ]:
X_train[:5]

array([[-1.44075296, -0.43531947, -1.36208497, -1.1391179 ,  0.78057331,
         0.71892128,  2.82313451, -0.11914956,  1.09266219,  2.45817261,
        -0.26380039, -0.01605246, -0.47041357, -0.47476088,  0.83836493,
         3.25102691,  8.43893667,  3.39198733,  2.62116574,  2.06120787,
        -1.23286131, -0.47630949, -1.24792009, -0.97396758,  0.72289445,
         1.18673232,  4.67282796,  0.9320124 ,  2.09724217,  1.88645014],
       [ 1.97409619,  1.73302577,  2.09167167,  1.85197292,  1.319843  ,
         3.42627493,  2.01311199,  2.66503199,  2.1270036 ,  1.55839569,
         0.80531919, -0.81268678,  0.75195659,  0.87716951, -0.89605315,
         1.18122247,  0.18362761,  0.60059598, -0.31771686,  0.52963649,
         2.17331385,  1.3112795 ,  2.08161691,  2.1374055 ,  0.76192793,
         3.26560084,  1.92862053,  2.6989469 ,  1.89116053,  2.49783848],
       [-1.39998202, -1.24962228, -1.34520926, -1.10978518, -1.33264483,
        -0.30735463, -0.36555756, -0.69650228,  1

### Display Scaled Training Data
This cell shows the first 5 rows of the standardized training data.

In [ ]:
Y_train.head()

,diagnosis
68,B
181,M
63,B
248,B
60,B


### Display Target Training Data
This cell shows the first 5 target labels for the training data.

In [ ]:
# Label encode

encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
Y_test = encoder.transform(Y_test)

### Encode Target Labels
This cell uses `LabelEncoder` to convert categorical target labels ('M' and 'B' for Malignant and Benign) into numerical format (0 and 1).

In [ ]:
Y_train[:5]

array([0, 1, 0, 0, 0])

### Display Encoded Target Labels
This cell shows the first 5 encoded target labels for the training data.

## Numpy arrays to PyTorch tensors

In [ ]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
Y_train_tensor = torch.from_numpy(Y_train.astype(np.float32))
Y_test_tensor = torch.from_numpy(Y_test.astype(np.float32))

### Convert to PyTorch Tensors
This cell converts the NumPy arrays of features and labels into PyTorch tensors. This is a necessary step before feeding the data into a PyTorch neural network model.

In [ ]:
X_train_tensor.shape

torch.Size([455, 30])

### Display Shape of Feature Tensor
This cell displays the shape of the training features tensor, indicating the number of samples and features.

In [ ]:
Y_train_tensor.shape

torch.Size([455])

### Display Shape of Target Tensor
This cell displays the shape of the training labels tensor.

## Let's now start with Neural Network

### Define the model

In [ ]:
class MySimpleNN():

    def __init__(self, X):

        # 30 x 1 - weight matrix
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float32, requires_grad=True) # Changed to torch.float32
        self.bias = torch.zeros(1, dtype=torch.float32, requires_grad=True) # Changed to torch.float32

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y):
        # clamp the prediction to avoid log(0)
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

        # calculate loss
        loss = - (y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()
        return loss

### Model Initialization and Definition
This cell defines a simple neural network class `MySimpleNN`. It includes:
- `__init__`: Initializes the weights and bias of the single-layer neural network.
- `forward`: Implements the forward pass, calculating the output using a sigmoid activation function.
- `loss_function`: Calculates the binary cross-entropy loss, clamping predictions to avoid `log(0)`.

### Important Parameters

In [ ]:
learning_rate = 0.1
epochs = 30

### Hyperparameters
This cell sets the `learning_rate` for the optimizer and the number of `epochs` for training the neural network.

### Training Pipeline

In [ ]:
# create model
model = MySimpleNN(X_train_tensor)
print(model.weights)
print(model.bias)

tensor([[0.8758],
        [0.9882],
        [0.9631],
        [0.0804],
        [0.4938],
        [0.2637],
        [0.8440],
        [0.6046],
        [0.9064],
        [0.0241],
        [0.0895],
        [0.4289],
        [0.6259],
        [0.9297],
        [0.2632],
        [0.1640],
        [0.6248],
        [0.6922],
        [0.3465],
        [0.6914],
        [0.5641],
        [0.5290],
        [0.7687],
        [0.9140],
        [0.6676],
        [0.9728],
        [0.9308],
        [0.1399],
        [0.5051],
        [0.6729]], requires_grad=True)
tensor([0.], requires_grad=True)


### Initialize Model
This cell creates an instance of the `MySimpleNN` model and prints the initial random weights and zero bias.

In [ ]:
# To train the model we need
# Perform in a Loop (epoch)
for epoch in range(epochs):
    # forward pass
    y_pred = model.forward(X_train_tensor)
    # print(y_pred)

    # calculate loss
    loss = model.loss_function(y_pred, Y_train_tensor)


    # backward pass
    loss.backward()

    # update parameters
    with torch.no_grad():
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad

    # zero gradiants
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    # print loss
    print(f"Epoch: {epoch + 1}, Loss: {loss.item()}")


Epoch: 1, Loss: 4.266100883483887
Epoch: 2, Loss: 4.151788234710693
Epoch: 3, Loss: 4.032031536102295
Epoch: 4, Loss: 3.910554885864258
Epoch: 5, Loss: 3.786891460418701
Epoch: 6, Loss: 3.660818338394165
Epoch: 7, Loss: 3.532524347305298
Epoch: 8, Loss: 3.4005072116851807
Epoch: 9, Loss: 3.2626068592071533
Epoch: 10, Loss: 3.1167635917663574
Epoch: 11, Loss: 2.9682798385620117
Epoch: 12, Loss: 2.8123130798339844
Epoch: 13, Loss: 2.643609046936035
Epoch: 14, Loss: 2.469761610031128
Epoch: 15, Loss: 2.2919692993164062
Epoch: 16, Loss: 2.11923885345459
Epoch: 17, Loss: 1.9468355178833008
Epoch: 18, Loss: 1.776962161064148
Epoch: 19, Loss: 1.6128756999969482
Epoch: 20, Loss: 1.4548674821853638
Epoch: 21, Loss: 1.3096790313720703
Epoch: 22, Loss: 1.180069088935852
Epoch: 23, Loss: 1.0683964490890503
Epoch: 24, Loss: 0.9762586951255798
Epoch: 25, Loss: 0.9040815830230713
Epoch: 26, Loss: 0.8506405353546143
Epoch: 27, Loss: 0.8130192756652832
Epoch: 28, Loss: 0.7873449921607971
Epoch: 29, Los

### Training Loop
This cell contains the main training loop:
- For each epoch, it performs a forward pass to get predictions.
- It calculates the loss using the defined `loss_function`.
- It performs a backward pass to compute gradients.
- It updates the model's weights and bias using the gradients and learning rate.
- It resets the gradients to zero for the next iteration.
- It prints the loss for each epoch.

In [ ]:
print(model.weights)
print(model.bias)

tensor([[ 0.2720],
        [ 0.4670],
        [ 0.3337],
        [-0.5086],
        [ 0.0318],
        [-0.4713],
        [ 0.0600],
        [-0.1756],
        [ 0.3750],
        [-0.2224],
        [-0.4452],
        [ 0.2191],
        [ 0.0910],
        [ 0.4596],
        [ 0.1109],
        [-0.3794],
        [ 0.1353],
        [ 0.0916],
        [ 0.1561],
        [ 0.3097],
        [-0.0948],
        [-0.0176],
        [ 0.0876],
        [ 0.2872],
        [ 0.1767],
        [ 0.2763],
        [ 0.1732],
        [-0.6541],
        [ 0.0244],
        [ 0.1570]], requires_grad=True)
tensor([-0.1433], requires_grad=True)


### Final Model Parameters
This cell prints the final learned weights and bias of the model after training.

### Evaluation

In [ ]:
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred_class = y_pred.round()
    accuracy = (y_pred_class == Y_test_tensor).float().mean()
    print(f"Accuracy: {accuracy.item()}")



Accuracy: 0.5215451121330261


### Evaluate Model Accuracy
This cell evaluates the trained model on the test set:
- It performs a forward pass on the test data.
- It rounds the predictions to get binary class predictions.
- It calculates the accuracy by comparing predicted classes with actual test labels and prints the result.